In [1]:
import sys, platform, subprocess

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(res.stdout if res.returncode == 0 else "[ERROR] No GPU allocated! Check Runtime settings.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Mon Oct 19 18:40:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                 

In [3]:
!pip install --quiet vllm huggingface_hub fastapi uvicorn httpx pydantic anyio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.5/326.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.7/504.7 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6

In [4]:
import os, hashlib
from huggingface_hub import snapshot_download

MODEL_REPO = "Qwen/Qwen2.5-1.5B-Instruct"
PINNED_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
TARGET_DIR = "/content/models/qwen2.5:1.5b"
EXPECTED_SHA256 = "dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee"

print(f"Fetching {MODEL_REPO} at pinned revision {PINNED_REVISION}...")
snapshot_download(
    repo_id=MODEL_REPO,
    revision=PINNED_REVISION,
    local_dir=TARGET_DIR,
    local_dir_use_symlinks=False
)

# Verify model.safetensors SHA-256
hasher = hashlib.sha256()
with open(os.path.join(TARGET_DIR, "model.safetensors"), "rb") as f:
    while chunk := f.read(1024 * 1024):
        hasher.update(chunk)
assert hasher.hexdigest() == EXPECTED_SHA256, "Upstream checksum mismatch!"
print("[PASS] Checksum verified: dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee")

# Confirm Apache 2.0 License
with open(os.path.join(TARGET_DIR, "LICENSE"), "r", encoding="utf-8") as f:
    lic = f.read()
assert "Apache License" in lic and "Version 2.0" in lic, "License is not Apache 2.0!"
print("[PASS] Apache 2.0 License confirmed.")


Fetching Qwen/Qwen2.5-1.5B-Instruct at pinned revision 989aa7980e4cf806f80c7fef2b1adb7bc71aa306...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[PASS] Checksum verified: dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee
[PASS] Apache 2.0 License confirmed.


In [5]:
import subprocess, time, httpx

log_out = open("/content/vllm.log", "w")
vllm_proc = subprocess.Popen([
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", TARGET_DIR,
    "--served-model-name", "qwen2.5:1.5b",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--max-model-len", "4096",
    "--max-num-seqs", "2",
    "--gpu-memory-utilization", "0.85"
], stdout=log_out, stderr=subprocess.STDOUT)

print("Booting vLLM engine... Waiting for health endpoint...")
for _ in range(60):
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=1.0).status_code == 200:
            print("[PASS] vLLM server is healthy and ready!")
            break
    except Exception:
        pass
    time.sleep(2)


Booting vLLM engine... Waiting for health endpoint...


In [6]:
import httpx, time

client = httpx.Client(base_url="http://127.0.0.1:8000/v1", timeout=30.0)

# Check /v1/models
models_resp = client.get("/models").json()
model_ids = [m["id"] for m in models_resp.get("data", [])]
print(f"Available models: {model_ids}")
assert "qwen2.5:1.5b" in model_ids, "Alias mismatch!"

# Test chat completion
t0 = time.perf_counter()
resp = client.post("/chat/completions", json={
    "model": "qwen2.5:1.5b",
    "messages": [
        {"role": "system", "content": "You are AeroGrid AI assistant."},
        {"role": "user", "content": "State the safety rules for high-voltage grid maintenance in 2 sentences."}
    ],
    "max_tokens": 64,
    "temperature": 0.2
}).json()
elapsed = time.perf_counter() - t0

print(f"\n[PASS] Response received in {elapsed:.2f}s:")
print(resp["choices"][0]["message"]["content"])

# Check GPU VRAM allocation
res = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(f"\nAllocated VRAM: {res.stdout.strip()}")


Available models: ['qwen2.5:1.5b']

[PASS] Response received in 1.56s:
High-voltage grid maintenance requires strict adherence to safety protocols, including wearing appropriate personal protective equipment (PPE), maintaining a safe distance from energized parts, and having emergency response plans in place.

Allocated VRAM: 12335 MiB, 15360 MiB


In [ ]:
# Stage AeroGrid AI backend in Colab
import zipfile, os, sqlite3, secrets

if not os.path.exists('/content/app'):
    if os.path.exists('/content/aerogrid_colab_bundle.zip'):
        with zipfile.ZipFile('/content/aerogrid_colab_bundle.zip', 'r') as z:
            z.extractall('/content')
        print('[PASS] Unpacked /content/aerogrid_colab_bundle.zip successfully!')
    else:
        print('Please upload aerogrid_colab_bundle.zip from project root:')
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded:
            if fn.endswith('.zip'):
                with zipfile.ZipFile(fn, 'r') as z:
                    z.extractall('/content')
                print(f'[PASS] Unpacked {fn} successfully!')

# 0. Clean any stale authentication databases so fresh ephemeral credentials apply
import glob
for stale in glob.glob("/content/auth.sqlite3*"):
    try:
        os.remove(stale)
    except Exception:
        pass

# 1. Initialize disposable sample_db.sqlite3 with synthetic employee directory
conn = sqlite3.connect('/content/sample_db.sqlite3')
conn.execute("CREATE TABLE IF NOT EXISTS employee_directory (employee_id TEXT PRIMARY KEY, display_name TEXT, department TEXT)")
conn.executemany("INSERT OR REPLACE INTO employee_directory VALUES (?, ?, ?)", [
    ('EMP-001', 'Alice Chen', 'Engineering'),
    ('EMP-002', 'Bob Smith', 'Engineering'),
    ('EMP-003', 'Carol Danvers', 'Product'),
    ('EMP-004', 'David Miller', 'Human Resources'),
    ('EMP-005', 'Emma Watson', 'Finance'),
])
conn.commit()
conn.close()
print('[PASS] Disposable sample_db.sqlite3 initialized with synthetic test records.')

# 2. Generate ephemeral runtime test credentials & application keys
runtime_user_pw = secrets.token_urlsafe(16)
runtime_admin_pw = secrets.token_urlsafe(16)

os.environ['SECRET_KEY'] = secrets.token_hex(32)
os.environ['DEV_AUTH_SIGNING_KEY'] = secrets.token_hex(32)
os.environ['AEROGRID_USER_PASSWORD'] = runtime_user_pw
os.environ['AEROGRID_ADMIN_PASSWORD'] = runtime_admin_pw
os.environ['DEV_DEMO_PASSWORD_ALICE'] = runtime_user_pw
os.environ['DEV_DEMO_PASSWORD_BOB'] = runtime_user_pw
os.environ['DEV_DEMO_PASSWORD_DAVID'] = runtime_user_pw
os.environ['DEV_DEMO_PASSWORD_EMMA'] = runtime_user_pw
os.environ['DEV_DEMO_ADMIN_PASSWORD'] = runtime_admin_pw
os.environ['DATABASE_URL'] = 'sqlite:////content/sample_db.sqlite3'
os.environ['AUTH_DB_PATH'] = '/content/auth.sqlite3'
os.environ['LLM_ENABLED'] = 'true'
os.environ['LLM_BASE_URL'] = 'http://127.0.0.1:8000/v1'
os.environ['LLM_ALLOWED_HOSTS'] = '["127.0.0.1", "localhost"]'
os.environ['LLM_MODEL'] = 'qwen2.5:1.5b'
os.environ['DOCUMENT_ROOTS'] = '["/content/sample_docs"]'
os.environ['ROLE_PERMISSION_MAP'] = json.dumps({"employee": ["chat:use", "documents:search", "database:read"], "engineer": ["chat:use", "documents:search", "database:read", "engineering:read"], "hr": ["chat:use", "documents:search", "database:read", "hr:read"], "analyst": ["chat:use", "database:read"], "admin": ["chat:use", "database:read", "admin:diagnostics", "users:read", "users:write", "audit:read"]})
os.environ['PORT'] = '8090'
print('[PASS] Ephemeral runtime test credentials and environment initialized safely.')

# 3. Install backend dependencies
!pip install --quiet 'langgraph' 'langchain-core' 'pydantic-settings' 'pymupdf' 'pyjwt[crypto]' 'sqlalchemy'


In [ ]:
import sys, subprocess, time, httpx, os

# Start AeroGrid FastAPI backend on port 8090 pointing to vLLM on port 8000
# (Note: Port 8080 is reserved by Google Colab's internal runtime proxy)
subprocess.run('fuser -k 8090/tcp 2>/dev/null || true', shell=True)
time.sleep(1)

backend_log = open('/content/aerogrid_backend.log', 'w')
backend_proc = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'app.main:create_app',
    '--factory', '--host', '127.0.0.1', '--port', '8090'
], stdout=backend_log, stderr=subprocess.STDOUT, env=dict(os.environ), cwd='/content')

print('Starting AeroGrid backend on port 8090... Waiting for /health/ready...')
backend_ready = False
for _ in range(30):
    try:
        if httpx.get('http://127.0.0.1:8090/health/ready', timeout=1.0).status_code == 200:
            backend_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if backend_ready:
    print('[PASS] AeroGrid FastAPI backend is live and healthy on port 8090!')
else:
    print('[FAIL] Backend failed to boot. Check logs:')
    with open('/content/aerogrid_backend.log') as f:
        print(f.read())


In [ ]:
# Full Integration Verification on Port 8090
import httpx, time, os

client = httpx.Client(base_url='http://127.0.0.1:8090', timeout=30.0)

# Retrieve runtime credentials from environment (never hardcoded in notebook)
admin_pw = os.environ['DEV_DEMO_ADMIN_PASSWORD']
alice_pw = os.environ['DEV_DEMO_PASSWORD_ALICE']
emma_pw = os.environ['DEV_DEMO_PASSWORD_EMMA']

# 1. Health check
h_resp = client.get('/health/ready')
assert h_resp.status_code == 200, f'Health check failed: {h_resp.text}'
print('[PASS] 1. /health/ready -> 200 OK')

# 2. Authentication (admin & demo accounts)
r_admin = client.post("/api/v1/auth/token", json={"username": "admin.system", "password": admin_pw})
assert r_admin.status_code == 200, f"Admin auth failed ({r_admin.status_code}): {r_admin.text}"
admin_tok = r_admin.json()["access_token"]
admin_headers = {'Authorization': f'Bearer {admin_tok}'}
print('[PASS] 2. /api/v1/auth/token -> 200 OK (JWT issued)')

# 3. Session validation
sess_resp = client.get('/api/v1/session', headers=admin_headers)
assert sess_resp.status_code == 200 and sess_resp.json()['subject'] == 'admin.system'
print('[PASS] 3. /api/v1/session -> 200 OK (admin.system)')

# 4. Document ACL filtering: engineer permitted vs analyst forbidden (403)
r_alice_tok = client.post("/api/v1/auth/token", json={"username": "alice.chen", "password": alice_pw})
assert r_alice_tok.status_code == 200, f"Alice auth failed ({r_alice_tok.status_code}): {r_alice_tok.text}"
alice_tok = r_alice_tok.json()["access_token"]
r_alice = client.post('/api/v1/mcp/tools/search_documents', headers={'Authorization': f'Bearer {alice_tok}'}, json={'parameters': {'query': 'turbine'}})
assert r_alice.status_code == 200, f'Engineer document search failed: {r_alice.text}'

r_emma_tok = client.post("/api/v1/auth/token", json={"username": "emma.watson", "password": emma_pw})
assert r_emma_tok.status_code == 200, f"Emma auth failed ({r_emma_tok.status_code}): {r_emma_tok.text}"
emma_tok = r_emma_tok.json()["access_token"]
r_emma = client.post('/api/v1/mcp/tools/search_documents', headers={'Authorization': f'Bearer {emma_tok}'}, json={'parameters': {'query': 'turbine'}})
assert r_emma.status_code == 403, f'Expected 403 for analyst, got {r_emma.status_code}'
print('[PASS] 4. Document ACL verification: Engineer 200 OK, Analyst 403 Forbidden')

# 5. Database catalog query
r_db = client.post('/api/v1/mcp/tools/query_database', headers=admin_headers, json={'parameters': {'query_id': 'directory.by_department', 'parameters': {'department': 'Engineering'}}})
assert r_db.status_code == 200, f'Database catalog query failed: {r_db.text}'
print('[PASS] 5. Database catalog query -> 200 OK')

# 6. Real chat completion routed through FastAPI to vLLM on port 8000
t0 = time.perf_counter()
r_chat = client.post('/api/v1/chat', headers=admin_headers, json={'message': 'State the safety rules for high-voltage grid maintenance in 2 sentences.'})
dur = time.perf_counter() - t0
assert r_chat.status_code == 200, f'Chat failed with status {r_chat.status_code}: {r_chat.text}'
print(f'[PASS] 6. /api/v1/chat -> 200 OK via vLLM qwen2.5:1.5b in {dur:.2f}s:')
print('   Response:', r_chat.json()['response'][:140], '...')


In [ ]:
# Run Mixed-Workload Benchmark against Port 8090 (10, 25, 50 virtual users)
!python scripts/benchmark_e2e_workload.py \
    --base-url http://127.0.0.1:8090 \
    --users 10,25,50 \
    --duration 15 \
    --skip-burst \
    --json-out /content/benchmark_colab_full_backend_measured.json


In [ ]:
# Parse and display Colab full-backend measured results
import json

with open('/content/benchmark_colab_full_backend_measured.json') as f:
    results = json.load(f)

print('=' * 80)
print(f"Colab Full-Backend Benchmark: {results.get('model_name')}")
print(f"Base URL: {results.get('base_url')} | Timestamp: {results.get('timestamp')}")
print('=' * 80)
for r in results.get('mixed_results', []):
    u = r['virtual_users']
    tot = r['total_requests']
    rps = r['throughput_rps']
    dur = r['duration_seconds']
    sc = r['status_codes']
    nlp95 = r['non_llm_p95_ms']
    cp50 = r['chat_p50_ms']
    cp95 = r['chat_p95_ms']
    print(f'Users: {u:2d} | Reqs: {tot:4d} | RPS: {rps:5.1f} | Duration: {dur:5.1f}s | Non-LLM p95: {nlp95:6.1f}ms | Chat p50: {cp50:7.1f}ms | Codes: {sc}')
